# 🚀 ACR-AGI-3 Kaggle Submission Notebook

メタスキル基盤（視覚ゲシュタルト直感・サブゴール分解・自己修復診断・動的スキルスコープ）による自律ゲームプレイ推論パイプライン。

### 📌 実行条件・制約
- **完全オフライン環境** (Internet: Disabled, `local_files_only=True`)
- **実行時間制限**: 最大 9 時間
- **ハードウェア**: Kaggle GPU (T4 / P100 / RTX Pro 6000)
- **出力**: カレントディレクトリ直下に `submission.json` を出力

In [ ]:
import sys
import os
import json
import time
from pathlib import Path

import numpy as np
import torch

print("=== System Environment ===")
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# === ライブラリパス解決 ===
# Kaggle Notebook の場合、リポジトリコードを作業ディレクトリまたは /kaggle/working/src に配置
repo_candidates = [
    Path("/kaggle/working/src"),
    Path("/kaggle/working/acr-agi3-edd-agent/src"),
    Path("/kaggle/input/acr-agi3-source/src"),
    Path("src"),
    Path("../src"),
]
for p in repo_candidates:
    if p.exists() and str(p.resolve()) not in sys.path:
        sys.path.insert(0, str(p.resolve()))
        print(f"✅ Added to sys.path: {p.resolve()}")

from acr_agi3.submission.path_resolver import ModelPathResolver
from acr_agi3.submission.entrypoint import KaggleSubmissionPipeline, run_submission
from acr_agi3.agent.orchestrator import ARCOrchestrator
from acr_agi3.game.env import Action

print("✅ acr_agi3 packages imported successfully!")

In [ ]:
# === モデル & データパスの検出 ===
model_path = ModelPathResolver.resolve_model_path()
data_dir = ModelPathResolver.resolve_data_dir()

print(f"🧠 Detected Local LLM Model Path: {model_path}")
print(f"📂 Detected Challenges Data Directory: {data_dir}")

# 課題ファイルの特定
challenge_candidates = [
    Path("/kaggle/input/arc-prize-2026-arc-agi-3/arc-agi_test_challenges.json"),
    data_dir / "arc-agi_test_challenges.json",
    data_dir / "test_challenges.json",
    Path("data/test_challenges.json"),
]
target_challenge_file = None
for c in challenge_candidates:
    if c.exists():
        target_challenge_file = c
        break

print(f"🎯 Selected Challenge File: {target_challenge_file}")

In [ ]:
# === Kaggle リーダーボード推論パイプラインの実行 ===
output_submission_path = Path("submission.json")

pipeline = KaggleSubmissionPipeline(
    model_path=model_path,
    max_steps_per_task=50,
    time_limit_per_task_sec=60.0,
)

if target_challenge_file and target_challenge_file.exists():
    print(f"▶️ Running submission on {target_challenge_file}...")
    results = pipeline.run_on_challenges(
        challenges_source=target_challenge_file,
        output_submission_path=output_submission_path,
    )
else:
    print("⚠️ Challenge file not found. Creating sample mock environment for smoke check...")
    # スモークテスト用モックデータ
    mock_challenges = {
        "sample_task_01": {
            "grid_shape": [10, 10],
            "initial_player_pos": [1, 1],
            "goal_pos": [8, 8],
            "walls": [[5, 0], [5, 1], [5, 2], [5, 3], [5, 4], [5, 5], [5, 6], [5, 7]],
            "hazards": [[3, 3]],
        }
    }
    results = pipeline.run_on_challenges(
        challenges_source=mock_challenges,
        output_submission_path=output_submission_path,
    )

In [ ]:
# === 提出ファイルのバリデーション検証 ===
assert output_submission_path.exists(), "submission.json was not created!"

with open(output_submission_path, "r", encoding="utf-8") as f:
    sub_data = json.load(f)

print("=== Submission Verification ===")
print(f"File Size: {output_submission_path.stat().st_size} bytes")
print(f"Total Tasks in Submission: {len(sub_data)}")

# 各タスクの内容チェック
for tid, entry in list(sub_data.items())[:3]:
    print(f"Task [{tid}]: Actions Count={len(entry.get('actions', []))}, Status={entry.get('status')}")

print("\n🎉 Submission ready for Kaggle Leaderboard!")